# Nexus: wlasny silnik w Colab lub Kaggle
Generowanie pojedynczego filmu: polska mowa + JoyVASA + LivePortrait. Bez HeyGen, bez lokalnej grafiki, bez serwera HTTP i tuneli.

**Uruchamiaj tylko w zdalnym runtime Colab/Kaggle, nie w VS Code ani w Colab local runtime.** Wybierz GPU T4 lub P100 i wlacz Internet. Dostepnosc GPU/limity zalezy od konta. Silnik instaluje osobny Python 3.11 przez uv, takze przy kernelu 3.13. Wymaga GPU zgodnego z CUDA 11.8; nie jest przeznaczony dla Blackwell.

Wgraj razem z notebookiem `render.py`, `download_models.py`, `requirements.txt` z katalogu avatar-cloud oraz obraz PNG/JPEG. Dla ruchu calego ciala wgraj tez `render_body.py`, `body_layers.py` i `requirements-body.txt`, a potem uruchom opcjonalne komorki na koncu. Colab: komorka upload ponizej. Kaggle: dodaj te pliki jako **prywatny** input (Add Input), nie publikuj zdjecia. Uzywaj wlasnej fikcyjnej postaci, nie podszywaj sie pod rzeczywiste osoby. Przestrzegaj zasad platformy; Colab zabrania deepfake i obchodzenia limitow.

Krotki render zweryfikowano w Colab na T4: MP4 768x960, 25 fps, 3.64 s, polska mowa i zmienne klatki. Kaggle pozostaje niezweryfikowane. Deformacja twarzy korzysta z oryginalnego PyTorch LivePortrait, gdyz standardowy ONNX Runtime nie obsluguje 5D GridSample. Bledy modeli/CUDA zatrzymuja wykonanie, nie tworza atrap. Glos espeak-ng jest syntetyczny, a animacja dotyczy glownie twarzy/glowy, nie calego ciala.

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
IS_KAGGLE = Path('/kaggle/working').is_dir() and bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
IS_COLAB = bool(os.environ.get('COLAB_RELEASE_TAG')) and Path('/content').is_dir()
if sys.platform != 'linux' or not (IS_KAGGLE or IS_COLAB):
    raise RuntimeError('Stop: wymagany zdalny runtime Colab/Kaggle. Lokalny komputer jest zablokowany.')
# Silnik uzyje oddzielnego Python 3.11, niezaleznie od wersji kernela.
subprocess.run(['nvidia-smi'], check=True)
ROOT = Path('/kaggle/working' if IS_KAGGLE else '/content') / 'nexus-avatar'
ROOT.mkdir(exist_ok=True)
print('Zdalne srodowisko:', 'Kaggle' if IS_KAGGLE else 'Colab', 'Katalog:', ROOT)


In [ ]:
if IS_COLAB:
    from google.colab import files
    from tempfile import TemporaryDirectory
    with TemporaryDirectory() as upload_dir:
        uploaded = files.upload(target_dir=upload_dir)
        for name, content in uploaded.items():
            (ROOT / Path(name).name).write_bytes(content)
else:
    INPUT_DIR = Path(input('Sciezka prywatnego inputu Kaggle, np. /kaggle/input/nexus-files: ').strip())
    if not INPUT_DIR.is_dir() or not INPUT_DIR.resolve().is_relative_to(Path('/kaggle/input')):
        raise ValueError('Wybierz katalog z /kaggle/input')
    for path in INPUT_DIR.iterdir():
        if path.is_file() and (path.suffix.lower() in ('.png', '.jpg', '.jpeg', '.json', '.mp4', '.mkv', '.avi') or path.name in ('render.py', 'download_models.py', 'requirements.txt', 'render_body.py', 'body_layers.py', 'requirements-body.txt', 'render_expression.py', 'expression_edit.py', 'expression_references.py', 'scene_layers.py', 'nexus_render_engine.py', 'compose_scene.py', 'reference_pack.py', 'render_rig.py', 'blender_rig.py', 'rig_motion.py', 'verify_rig.py', 'batch_job.py', 'worker.py')):
            shutil.copy2(path, ROOT / path.name)
for name in ('render.py', 'download_models.py', 'requirements.txt'):
    if not (ROOT / name).is_file():
        raise FileNotFoundError('Wgraj brakujacy plik: ' + name)
print('Pliki:', [p.name for p in ROOT.iterdir() if p.is_file()])


In [ ]:
def checked(args, **kwargs):
    result = subprocess.run(args, capture_output=True, text=True, **kwargs)
    print(result.stdout[-6000:]); print(result.stderr[-6000:])
    result.check_returncode()
checked(['apt-get', 'update'])
checked(['apt-get', 'install', '-y', 'ffmpeg', 'espeak-ng', 'git', 'libgl1', 'libglib2.0-0', 'build-essential'])
# Osobne srodowisko chroni pakiety dostarczone przez Colab/Kaggle.
VENV = ROOT / 'venv'
BOOTSTRAP = ROOT / 'bootstrap'
checked([sys.executable, '-m', 'pip', 'install', '--target', str(BOOTSTRAP), 'uv==0.8.22'])
UV = str(BOOTSTRAP / 'bin/uv')
checked([UV, 'venv', '--python', '3.11', str(VENV), '--seed'])
PYTHON = str(VENV / 'bin/python')
checked([PYTHON, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt')])
checked([PYTHON, '-m', 'pip', 'check'])
ENGINE = ROOT / 'FasterLivePortrait'
if not ENGINE.exists():
    checked(['git', 'clone', 'https://github.com/warmshao/FasterLivePortrait.git', str(ENGINE)])
checked(['git', '-C', str(ENGINE), 'checkout', '8aad3602177547aaa5e4beec0c3ef5b7944e7a1f'])
ENV = {**os.environ, 'NEXUS_CLOUD_WORKER': '1', 'NEXUS_ENGINE_DIR': str(ENGINE)}
checked([PYTHON, str(ROOT / 'download_models.py')], env=ENV)
print('Modele i silnik przygotowane w chmurze.')


In [ ]:
import uuid
PORTRAIT_NAME = input('Nazwa wgranego obrazu PNG/JPEG: ').strip()
PORTRAIT = ROOT / PORTRAIT_NAME
if Path(PORTRAIT_NAME).name != PORTRAIT_NAME or not PORTRAIT.is_file() or PORTRAIT.suffix.lower() not in ('.png', '.jpg', '.jpeg'):
    raise ValueError('Wybierz wgrany obraz PNG/JPEG z katalogu notebooka')
TEXT = input('Krotka polska wypowiedz do testu (maks. 300 znakow): ').strip()
if not TEXT or len(TEXT) > 300:
    raise ValueError('Test wymaga 1-300 znakow')
JOB = ROOT / ('job-' + str(uuid.uuid4()))
JOB.mkdir()
shutil.copy2(PORTRAIT, JOB / ('portrait' + PORTRAIT.suffix.lower()))
(JOB / 'script.txt').write_text(TEXT, encoding='utf-8')
checked([PYTHON, str(ROOT / 'render.py'), str(JOB)], env=ENV, timeout=1800)
VIDEO = JOB / 'video.mp4'
if not VIDEO.is_file():
    raise RuntimeError('Brak wynikowego filmu. Nie zglaszamy sukcesu.')
print('Film:', VIDEO)


In [ ]:
from IPython.display import Video, display, FileLink
display(Video(str(VIDEO), embed=True, width=384))
print('Sprawdz ruch twarzy, synchronizacje ust, dzwiek i caly kadr przed uznaniem renderu za poprawny.')
if IS_COLAB:
    files.download(str(VIDEO))
else:
    OUTPUT = Path('/kaggle/working/nexus-preview.mp4')
    shutil.copy2(VIDEO, OUTPUT)
    display(FileLink(str(OUTPUT)))
    print('Pobierz nexus-preview.mp4 z Output. Nie publikuj notebooka ani wynikow ze zdjeciem prywatnym.')


## Zadanie z aplikacji Nexus
W aplikacji rozwin Silnik Colab, wpisz wypowiedz i pobierz JSON. Wgraj ten plik oraz batch_job.py i worker.py z avatar-cloud. Uruchom komorke ponizej po instalacji srodowiska; poprzedni reczny render nie jest wymagany. Wynik sprawdz, pobierz i wczytaj w Nexusie przyciskiem Wczytaj film z Colab. To przenoszenie plikow, nie staly serwer ani tunel. Zdjecie i tekst sa wewnatrz JSON — nie publikuj go.


In [ ]:
from IPython.display import Video, display, FileLink
for name in ('batch_job.py', 'worker.py'):
    if not (ROOT / name).is_file():
        raise FileNotFoundError('Wgraj brakujacy plik: ' + name)
REQUEST_NAME = input('Nazwa JSON pobranego z Nexusa: ').strip()
if Path(REQUEST_NAME).name != REQUEST_NAME or not (ROOT / REQUEST_NAME).is_file() or Path(REQUEST_NAME).suffix.lower() != '.json':
    raise ValueError('Wybierz wgrany plik JSON z katalogu notebooka')
checked([PYTHON, str(ROOT / 'batch_job.py'), str(ROOT / REQUEST_NAME), str(ROOT)], env=ENV, timeout=1900)
import json
REQUEST_ID = json.loads((ROOT / REQUEST_NAME).read_text(encoding='utf-8'))['requestId']
BATCH_VIDEO = ROOT / ('batch-' + REQUEST_ID) / 'video.mp4'
display(Video(str(BATCH_VIDEO), embed=True, width=384))
print('Sprawdz tozsamosc, ruch ust i polska mowe przed wczytaniem filmu do Nexusa.')
if IS_COLAB:
    files.download(str(BATCH_VIDEO))
else:
    BATCH_OUTPUT = Path('/kaggle/working') / ('nexus-' + REQUEST_ID + '.mp4')
    shutil.copy2(BATCH_VIDEO, BATCH_OUTPUT)
    display(FileLink(str(BATCH_OUTPUT)))


## Opcjonalnie: rece, nogi i tulow — LTX image-to-video
Ten tryb generuje ruch calego obrazu z portretu, a nie tylko twarzy. Osobny Python chroni dzialajacy LivePortrait. Pierwsza instalacja i modele wymagaja dodatkowej pamieci dysku oraz czasu. Dla portretu glowy i dloni wpisz opis delikatnego ruchu glowy/nadgarstka zamiast chodzenia. Model LTX 0.9.5 generuje obraz z dluzszym bokiem 768 px, przy zachowaniu proporcji i paddingu zamiast obcinania. Model moze zmienic detale lub rozmyc obraz — trzeba obejrzec ostrosc i kazda poruszana czesc, sama zmiana pikseli nie jest dowodem. Wczesniejsze proby calego ciala 384x480 nie spelnily wymagan ruchu nog i jakosci. Polska mowa jest dolaczana do filmu (do 4.04 s), ale ten tryb **nie synchronizuje ust z tekstem**. Nie laczymy automatycznie dwoch niezgodnych animacji. To nie staly backend aplikacji.


In [ ]:
if sys.platform != 'linux' or not (IS_COLAB or IS_KAGGLE):
    raise RuntimeError('Wymagany zdalny runtime Colab/Kaggle')
for name in ('render.py', 'render_body.py', 'body_layers.py', 'requirements-body.txt'):
    if not (ROOT / name).is_file():
        raise FileNotFoundError('Wgraj plik: ' + name)
BODY_VENV = ROOT / 'body-venv'
if not (BODY_VENV / 'bin/python').exists():
    checked([UV, 'venv', '--python', '3.11', str(BODY_VENV), '--seed'])
BODY_PYTHON = str(BODY_VENV / 'bin/python')
checked([BODY_PYTHON, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements-body.txt')], timeout=1200)
checked([BODY_PYTHON, '-m', 'pip', 'check'])


In [ ]:
import uuid
if not PORTRAIT.is_file():
    raise FileNotFoundError('Najpierw wybierz portret w komorce render')
BODY_JOB = ROOT / ('body-' + str(uuid.uuid4()))
BODY_JOB.mkdir()
shutil.copy2(PORTRAIT, BODY_JOB / ('portrait' + PORTRAIT.suffix.lower()))
(BODY_JOB / 'script.txt').write_text(TEXT, encoding='utf-8')
MOTION_TEXT = input('Opis ruchu po angielsku (Enter = domyslne kroki i ruch ramion): ').strip()
if MOTION_TEXT:
    (BODY_JOB / 'motion.txt').write_text(MOTION_TEXT, encoding='utf-8')
checked([BODY_PYTHON, '-u', str(ROOT / 'render_body.py'), str(BODY_JOB)], env=ENV, timeout=2400)
BODY_VIDEO = BODY_JOB / 'video.mp4'
if not BODY_VIDEO.is_file():
    raise RuntimeError('Brak filmu z ruchem ciala')
display(Video(str(BODY_VIDEO), embed=True, width=384))
print('Obejrzyj osobno rece, nogi i tulow. Glos dolaczony, bez lip-sync.')
if IS_COLAB:
    files.download(str(BODY_VIDEO))
else:
    BODY_OUTPUT = Path('/kaggle/working/nexus-body-preview.mp4')
    shutil.copy2(BODY_VIDEO, BODY_OUTPUT)
    display(FileLink(str(BODY_OUTPUT)))


## Eksperymentalnie: wspolna edycja mimiki bez regenerowania dloni
Najpierw zaakceptuj film ruchu ciala. Wgraj `render_expression.py`, `expression_edit.py`, `scene_layers.py` i film maski grayscale (najlepiej lossless FFV1 MKV), o identycznej rozdzielczosci/FPS/liczbie klatek. Biale piksele maski oznaczaja edycje oczu/ust; czarne zachowuja odpowiadajaca klatke ruchu ciala. Maska moze zmieniac sie w czasie, np. oczy tylko przez kilka klatek. Wybierz rzeczywisty obszar twarzy; maska poza nim jest odrzucana. To nie osobna doklejona glowa: VACE edytuje ten sam film, a niezamaskowane piksele dloni/ciala sa przywracane przed kodowaniem. Kompresja MP4 nadal zmienia nieznacznie piksele. Wymagane komorki cloud-guard i body-install; dodatkowe modele pobierane tylko w chmurze. Ten prototyp wymaga kalibracji maski i stabilnej pozy glowy, nie daje automatycznego sledzenia ani lip-sync. Obejrzyj mimike i granice maski; samo zachowanie dloni nie potwierdza poprawnego mrugniecia.


In [ ]:
import json, uuid
if sys.platform != 'linux' or not (IS_COLAB or IS_KAGGLE):
    raise RuntimeError('Wymagany zdalny runtime Colab/Kaggle')
for name in ('render_expression.py', 'expression_edit.py', 'scene_layers.py'):
    if not (ROOT / name).is_file():
        raise FileNotFoundError('Wgraj plik: ' + name)
SOURCE_NAME = input('Wgrany zaakceptowany film ciala (Enter = ostatni BODY_VIDEO): ').strip()
if SOURCE_NAME:
    if Path(SOURCE_NAME).name != SOURCE_NAME:
        raise ValueError('Wybierz nazwe wgranego filmu, bez sciezki')
    EXPRESSION_SOURCE = ROOT / SOURCE_NAME
else:
    if 'BODY_VIDEO' not in globals():
        raise ValueError('Brak poprzedniego filmu; podaj nazwe wgranego zaakceptowanego filmu ciala')
    EXPRESSION_SOURCE = BODY_VIDEO
MASK_NAME = input('Nazwa wgranego filmu maski grayscale: ').strip()
if not MASK_NAME or Path(MASK_NAME).name != MASK_NAME:
    raise ValueError('Wybierz nazwe wgranego filmu maski')
FACE_REGION = json.loads(input('Skalibrowany obszar twarzy [x,y,width,height]: ').strip())
EXPRESSION_JOB = ROOT / ('expression-' + str(uuid.uuid4()))
EXPRESSION_JOB.mkdir()
(EXPRESSION_JOB / 'expression-edit.json').write_text(json.dumps({'sourceClip': str(EXPRESSION_SOURCE), 'maskClip': str(ROOT / MASK_NAME), 'faceRegion': FACE_REGION}), encoding='utf-8')
EXPRESSION_TEXT = input('Opis mimiki po angielsku (Enter = szybki blink i maly usmiech): ').strip()
if EXPRESSION_TEXT:
    (EXPRESSION_JOB / 'expression.txt').write_text(EXPRESSION_TEXT, encoding='utf-8')
checked([BODY_PYTHON, '-u', str(ROOT / 'render_expression.py'), str(EXPRESSION_JOB)], env=ENV, timeout=3600)
EXPRESSION_VIDEO = EXPRESSION_JOB / 'expression.mp4'
display(Video(str(EXPRESSION_VIDEO), embed=True, width=384))
print('Sprawdz oczy, usta i szwy maski; rece/cialo pochodza z tej samej osi czasu. Bez lip-sync.')
if IS_COLAB:
    files.download(str(EXPRESSION_VIDEO))
else:
    display(FileLink(str(EXPRESSION_VIDEO)))


## Opcjonalnie: Nexus Render Engine - kolejka przygotowanych zadan
Wgraj nexus_render_engine.py i adaptery wybrane w render-plan.json. Przygotuj nowy katalog batch z planem version:1 oraz jobs:[{engine,directory}]; kazde directory musi byc osobnym bezposrednim podkatalogiem. Silniki: faster-liveportrait, flashhead, ltx-body, vace-expression, scene-composer. MuseTalk, Wan2.2 i vace-body nie sa jeszcze zintegrowane i zostana jawnie odrzucone. Zadania maja osobne wejscia zgodne z dokumentacja adapterow; kolejka NIE laczy automatycznie twarzy z cialem. VACE wymaga zaakceptowanego sourceClip i maski tej samej osi czasu. Kontroler zapisuje render-state.json i nexus-render.log, nie nadpisuje wynikow ani historii. Status rendered wymaga nadal oceny wzrokowej. W razie bledu dalsze zadania sa blokowane; nie ma ukrytej zmiany rozdzielczosci ani lokalnego GPU. Osobne srodowiska ustaw przez NEXUS_RENDER_PYTHON_* zgodnie z DEPLOY.md.


In [ ]:
from pathlib import Path
import sys
if sys.platform != 'linux' or not (IS_COLAB or IS_KAGGLE):
    raise RuntimeError('Nexus Render Engine wymaga zdalnej chmury')
if not (ROOT / 'nexus_render_engine.py').is_file():
    raise FileNotFoundError('Wgraj nexus_render_engine.py')
BATCH_NAME = input('Nazwa przygotowanego katalogu batch z render-plan.json: ').strip()
BATCH_ROOT = (ROOT / BATCH_NAME).resolve()
if not BATCH_NAME or BATCH_ROOT.parent != ROOT.resolve() or not BATCH_ROOT.is_dir():
    raise ValueError('Wybierz istniejacy podkatalog runtime')
checked([PYTHON, '-u', str(ROOT / 'nexus_render_engine.py'), str(BATCH_ROOT)], env=ENV, timeout=58000)
print((BATCH_ROOT / 'render-state.json').read_text())
print('Rendered nie oznacza akceptacji mimiki, anatomii ani lip-sync; obejrzyj kazdy wynik.')


## Opcjonalnie: pakiet obrazow tej samej postaci
Wgraj `reference_pack.py` i kolaz 6 kolumn x 2 rzedy (kolejnosc: neutral, eyes_closed, wink, smile, frown, reach_front, profile_left, three_quarter, back_three_quarter, reach_close, reach_side, smile_eyes_closed). Silnik wycina 12 referencji i tworzy character.json. Role sa etykietami pozycji, nie rozpoznawaniem min. Dostarczone plansze kosmiczne 6x5 maja szablony cosmic-expressions (domyslna rola calm) i cosmic-storyboard (eye_contact); nie stosuj tych etykiet do innych plansz. Kadry scen/logo nie sa zrodlem twarzy. Opcja bottom odcina podany dolny pasek podpisu kazdego panelu, nie rozpoznaje tekstu ani numerow. Inne uklady obsluguje CLI --columns/--rows. Wybrana referencja sluzy jako zrodlo istniejacego silnika twarzy, nie jako film ruchu ani rekonstrukcja 3D. Sprawdz kadry, etykiety i rozdzielczosc przed uzyciem. Do mowienia wybierz neutral lub three_quarter z otwartymi oczami; mocny profil/zaslonieta twarz moze nie przejsc detekcji. Zachowaj chlopca jako glowna postac; kazdy import tworzy osobny pakiet.


In [ ]:
import json, uuid
from IPython.display import Video, display, FileLink
COLLAGE_NAME = input('Nazwa wgranego kolazu: ').strip()
if Path(COLLAGE_NAME).name != COLLAGE_NAME or not (ROOT / COLLAGE_NAME).is_file():
    raise ValueError('Wybierz wgrany kolaz')
REF_PACK = ROOT / ('character-' + str(uuid.uuid4()))
LAYOUT = input('Szablon: grid / cosmic-expressions / cosmic-storyboard: ').strip()
if LAYOUT not in ('grid', 'cosmic-expressions', 'cosmic-storyboard'):
    raise ValueError('Wybierz jawny szablon kolazu')
BOTTOM = int(input('Ile pikseli podpisu odciac z dolu kazdego panelu (0-64): ').strip())
checked([PYTHON, str(ROOT / 'reference_pack.py'), 'import', str(ROOT / COLLAGE_NAME), str(REF_PACK), '--layout', LAYOUT, '--bottom', str(BOTTOM)])
print((REF_PACK / 'character.json').read_text())
ROLE = input('Rola referencji z manifestu, np. neutral / calm / eye_contact: ').strip()
REF_JOB = ROOT / ('reference-face-' + str(uuid.uuid4()))
REF_JOB.mkdir()
checked([PYTHON, str(ROOT / 'reference_pack.py'), 'select', str(REF_PACK), ROLE, str(REF_JOB)])
REF_TEXT = input('Krotka polska wypowiedz (1-300 znakow): ').strip()
if not REF_TEXT or len(REF_TEXT) > 300:
    raise ValueError('Wymagana wypowiedz 1-300 znakow')
(REF_JOB / 'script.txt').write_text(REF_TEXT, encoding='utf-8')
checked([PYTHON, str(ROOT / 'render.py'), str(REF_JOB)], env=ENV, timeout=1800)
display(Video(str(REF_JOB / 'video.mp4'), embed=True, width=384))
PACK_ARCHIVE = shutil.make_archive(str(REF_PACK), 'zip', REF_PACK)
if IS_COLAB:
    files.download(PACK_ARCHIVE)
    files.download(str(REF_JOB / 'video.mp4'))
else:
    display(FileLink(PACK_ARCHIVE), FileLink(str(REF_JOB / 'video.mp4')))


## Opcjonalnie: kontrolowany szkielet 3D — prototyp
Wgraj `render_rig.py`, `blender_rig.py`, `rig_motion.py`, `verify_rig.py` oraz `render.py`. Blender jest instalowany tylko w chmurze, renderuje na CPU chmury (bez grafiki komputera). Postac to uproszczony chlopiec w niebieskiej bluzie, z 27 kosciami, oczami, ustami i palcami. NIE jest odtworzeniem chlopca lub bohaterki ze zdjec. Referencje sa osobnym pakietem, nie tekstura ani automatycznym rigiem. Kamera body/face; ruch idle/wave/walk/showcase. Showcase: machanie, potem kroki w miejscu. To proceduralne ruchy bez motion capture/IK; usta reaguja na energie audio, nie na fonemy. Zapis GLB zawiera skin i animacje, BLEND umozliwia dalsza edycje. To batch offline, nie integracja 3D w aplikacji ani staly serwer.


In [ ]:
import json, uuid, zipfile
from IPython.display import Video, display, FileLink
if sys.platform != 'linux' or not (IS_COLAB or IS_KAGGLE):
    raise RuntimeError('Szkielet renderujemy wylacznie w zdalnym runtime')
for name in ('render_rig.py', 'blender_rig.py', 'rig_motion.py', 'verify_rig.py', 'render.py'):
    if not (ROOT / name).is_file():
        raise FileNotFoundError('Wgraj plik: ' + name)
if not shutil.which('blender'):
    checked(['apt-get', 'install', '-y', 'blender'])
checked(['blender', '--version'])
RIG_CAMERA = input('Kamera body lub face: ').strip()
RIG_ACTION = input('Ruch idle, wave, walk lub showcase: ').strip()
RIG_TEXT = input('Krotka wypowiedz do 6 sekund: ').strip()
RIG_JOB = ROOT / ('rig-' + str(uuid.uuid4()))
RIG_JOB.mkdir()
(RIG_JOB / 'rig.json').write_text(json.dumps({'camera': RIG_CAMERA, 'action': RIG_ACTION}), encoding='utf-8')
(RIG_JOB / 'script.txt').write_text(RIG_TEXT, encoding='utf-8')
checked([PYTHON, '-u', str(ROOT / 'render_rig.py'), str(RIG_JOB)], env=ENV, timeout=2700)
print((RIG_JOB / 'export-verification.json').read_text())
display(Video(str(RIG_JOB / 'video.mp4'), embed=True, width=384))
RIG_ARCHIVE = RIG_JOB / 'nexus-rig.zip'
with zipfile.ZipFile(RIG_ARCHIVE, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in ('character.glb', 'character.blend', 'video.mp4', 'rig.json', 'rig-report.json', 'export-verification.json'):
        archive.write(RIG_JOB / name, name)
if IS_COLAB:
    files.download(str(RIG_ARCHIVE))
else:
    display(FileLink(str(RIG_ARCHIVE)))


## Independent FlashHead Lite cloud preview
Do not run all cells. This mode does not use the earlier LivePortrait environment. Select a remote L4/A100-or-newer GPU with at least 16 GiB VRAM, not T4. Upload setup_flashhead_cloud.py, render_flashhead.py, render.py and an authorized fictional portrait named nexus-boy.png into /content using Colab's Files panel. Model downloads and rendering consume remote Colab resources. This is batch MP4, not a permanent server or measured real-time stream. Save the result outside the temporary VM.


In [ ]:
import os, sys, subprocess
from pathlib import Path
if sys.platform != 'linux' or not os.environ.get('COLAB_RELEASE_TAG') or not Path('/content').is_dir():
    raise RuntimeError('Remote Colab runtime required')
os.environ['NEXUS_CLOUD_WORKER'] = '1'
os.environ['NEXUS_MODEL_DOWNLOAD_CONSENT'] = '1'
result = subprocess.run([sys.executable, '/content/setup_flashhead_cloud.py'], capture_output=True, text=True)
print(result.stdout[-8000:]); print(result.stderr[-8000:]); result.check_returncode()


In [ ]:
import os, sys, subprocess, shutil, uuid
from pathlib import Path
from IPython.display import Video, display
if sys.platform != 'linux' or not os.environ.get('COLAB_RELEASE_TAG'):
    raise RuntimeError('Remote Colab runtime required')
FLASH_ROOT = Path('/content/nexus-flashhead-lite')
FLASH_QUALITY = 'lite'  # Set 'pro' for the slower quality variant; upload setup_flashhead_models.py too.
if FLASH_QUALITY not in ('lite', 'pro'):
    raise ValueError('Quality must be lite or pro')
FLASH_PYTHON = FLASH_ROOT / '.venv/bin/python'
FLASH_JOB = FLASH_ROOT / ('preview-' + str(uuid.uuid4()))
if not FLASH_PYTHON.is_file() or not Path('/content/nexus-boy.png').is_file():
    raise RuntimeError('Complete setup and upload the portrait first')
FLASH_JOB.mkdir()
shutil.copy2('/content/nexus-boy.png', FLASH_JOB / 'portrait.png')
(FLASH_JOB / 'script.txt').write_text('Czesc, jestem Nexus. Milo cie widziec.', encoding='utf-8')
FLASH_ENV = dict(os.environ, NEXUS_CLOUD_WORKER='1', NEXUS_FLASHHEAD_DIR=str(FLASH_ROOT / 'engine'), NEXUS_FLASHHEAD_MODEL=FLASH_QUALITY, MPLBACKEND='Agg')
if FLASH_QUALITY == 'pro':
    FLASH_ENV['NEXUS_MODEL_DOWNLOAD_CONSENT'] = '1'
    subprocess.run([str(FLASH_PYTHON), '/content/setup_flashhead_models.py'], env=FLASH_ENV, check=True, timeout=1200)
with (FLASH_JOB / 'render.log').open('w') as log:
    subprocess.run([str(FLASH_PYTHON), '/content/render_flashhead.py', str(FLASH_JOB)], env=FLASH_ENV, check=True, timeout=1900, stdout=log, stderr=subprocess.STDOUT)
display(Video(str(FLASH_JOB / 'video.mp4'), embed=True, width=512))
print('Inspect identity, eye/head motion and lip timing. This is not a live deployment.')
from google.colab import files
files.download(str(FLASH_JOB / 'video.mp4'))
